# Fashion For Everyone - Clothing Segmentation Pipeline

**Pipeline:** Image → Grounding DINO detects clothing → SAM 2 segments it → Extracted clothing on white/transparent background

**Environment:** Google Colab (T4 GPU free tier is sufficient)

### Expected Input Structure
```
DATASET_ROOT/
  {category}/
      image_0.jpg, image_1.jpg, ...
      description.txt
```

### Output Structure
```
OUTPUT_ROOT/
  {category}/
      image_0_seg_0.jpg        (clothing on white bg)
      image_0_seg_0.png        (clothing, transparent bg)
      image_0_seg_0_mask.png   (binary mask)
      image_0_annotated.jpg    (visualization)
      metadata.json            (detection metadata)
```

---
## 1. Setup & Installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q torch torchvision --upgrade
!pip install -q transformers>=4.40.0 accelerate supervision>=0.21.0
!pip install -q sam2 pillow opencv-python-headless tqdm scikit-image

# Clone Grounded SAM 2 and install
!git clone -q https://github.com/IDEA-Research/Grounded-SAM-2.git
%cd Grounded-SAM-2

# Install SAM 2 (skip CUDA extension build if it fails - still works)
!SAM2_BUILD_CUDA=0 pip install -e . -q

# Install Grounding DINO
!pip install --no-build-isolation -e grounding_dino -q

# Download SAM 2 checkpoints
%cd checkpoints
!bash download_ckpts.sh
%cd /content

---
## 2. Imports & Environment Check

In [ ]:
import os
import sys
import json
import csv
import re
import shutil
from typing import Dict, List, Tuple, Optional
from pathlib import Path

import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
from IPython.display import display

from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

sys.path.insert(0, '/content/Grounded-SAM-2')
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

import supervision as sv
from skimage.morphology import convex_hull_image

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("All imports successful!")

---
## 3. Configuration

Edit the paths and category prompts below to match your setup.

In [ ]:
class Config:
    """Central configuration for the segmentation pipeline."""

    # --- Device ---
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # --- Paths (adjust to your Colab/Drive structure) ---
    # Expected input structure:
    #   DATASET_ROOT/
    #     tshirts/
    #       image_0.jpg
    #       image_1.jpg
    #       description.txt
    #     casual-shirts/
    #       image_0.jpg
    #       description.txt
    #       ...
    DATASET_ROOT = "/content/drive/MyDrive/fashion_dataset"
    OUTPUT_ROOT = "/content/drive/MyDrive/fashion_segmented-final"

    # --- Model paths ---
    SAM2_CHECKPOINT = "/content/Grounded-SAM-2/checkpoints/sam2.1_hiera_large.pt"
    SAM2_MODEL_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"
    GDINO_MODEL_ID = "IDEA-Research/grounding-dino-tiny"

    # --- Detection thresholds ---
    BOX_THRESHOLD = 0.30
    TEXT_THRESHOLD = 0.25

    # --- Clothing categories and their text prompts ---
    # IMPORTANT: Grounding DINO prompts must be lowercase and end with a period
    MENS_CLOTHING = {
        'tshirts':           't-shirt. crew neck top.',
        'casual-shirts':     'casual shirt. button-up shirt.',
        'formal-shirts':     'formal shirt. dress shirt.',
        'sweatshirts':       'sweatshirt. hoodie. pullover.',
        'sweaters':          'sweater. knit pullover.',
        'jackets':           'jacket.',
        'blazers':           'blazer.',
        'suits':             'suit.',
        'rain-jacket':       'rain jacket. waterproof jacket.',
        'jeans':             'jeans. denim trousers.',
        'casual-trousers':   'trousers. casual pants.',
        'formal-trousers':   'formal trousers. dress pants.',
        'shorts':            'shorts.',
        'trackpants':        'track pants. joggers.',
    }

    WOMENS_CLOTHING = {
        'women-kurtas-kurtis-suits': 'draped fabric. garment.',
        'ethnic-tops':               'ethnic top. embroidered blouse.',
        'saree':                     'draped fabric. garment.',
        'women-ethnic-wear':         'ethnic dress. garment.',
        'women-ethnic-bottomwear':   'ethnic bottom. palazzo. salwar.',
        'skirts-palazzos':           'skirt. palazzo.',
        'lehenga-choli':             'skirt. blouse.',
        'dupatta-shawl':             'scarf. shawl. stole.',
        'women-jackets':             'jacket.',
    }

    ALL_CATEGORIES = {**MENS_CLOTHING, **WOMENS_CLOTHING}

    # --- Output settings ---
    SAVE_VISUALIZATION = True
    SAVE_MASK = True
    BACKGROUND_COLOR = (255, 255, 255)
    SAVE_TRANSPARENT = True


config = Config()
print(f"Device: {config.DEVICE}")
print(f"Dataset root: {config.DATASET_ROOT}")
print(f"Output root: {config.OUTPUT_ROOT}")
print(f"Categories: {len(config.ALL_CATEGORIES)}")

---
## 4. Load Models

In [ ]:
def load_models(config: Config) -> dict:
    """
    Load Grounding DINO (HuggingFace) and SAM 2 models.
    """
    print(f"Using device: {config.DEVICE}")

    print("Loading Grounding DINO from HuggingFace...")
    gdino_processor = AutoProcessor.from_pretrained(config.GDINO_MODEL_ID)
    gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
        config.GDINO_MODEL_ID
    ).to(config.DEVICE)
    gdino_model.eval()

    print("Loading SAM 2.1 Large...")
    sam2_model = build_sam2(
        config.SAM2_MODEL_CFG,
        config.SAM2_CHECKPOINT,
        device=config.DEVICE,
    )
    sam2_predictor = SAM2ImagePredictor(sam2_model)

    print("Models loaded successfully!")

    return {
        'gdino_processor': gdino_processor,
        'gdino_model': gdino_model,
        'sam2_predictor': sam2_predictor,
    }


models = load_models(config)

---
## 5. Core Functions

Detection (Grounding DINO) → Segmentation (SAM 2) → Mask cleaning → Extraction

In [ ]:
def detect_clothing(
    image: Image.Image,
    text_prompt: str,
    models: dict,
    config: Config,
) -> Tuple[np.ndarray, np.ndarray, list]:
    """
    Use Grounding DINO to detect clothing items in the image.

    Returns:
        boxes: (N, 4) xyxy format
        scores: (N,)
        labels: list of str
    """
    processor = models['gdino_processor']
    model = models['gdino_model']

    inputs = processor(
        images=image, text=text_prompt, return_tensors="pt"
    ).to(config.DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        target_sizes=[image.size[::-1]],
    )[0]

    # Manual thresholding
    keep = results["scores"] >= config.BOX_THRESHOLD
    boxes = results["boxes"][keep].cpu().numpy()
    scores = results["scores"][keep].cpu().numpy()
    labels = [label for label, k in zip(results["labels"], keep) if k]

    return boxes, scores, labels

In [ ]:
def segment_with_sam2(
    image_np: np.ndarray,
    boxes: np.ndarray,
    models: dict,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Use SAM 2 to generate pixel-level masks from bounding boxes.

    Returns:
        masks: (N, H, W) boolean
        iou_scores: (N,)
    """
    predictor = models['sam2_predictor']
    predictor.set_image(image_np)

    if len(boxes) == 0:
        return np.array([]), np.array([])

    masks, iou_scores, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=boxes,
        multimask_output=False,
    )

    if masks.ndim == 4:
        masks = masks.squeeze(1)
    if iou_scores.ndim == 2:
        iou_scores = iou_scores.squeeze(1)

    if isinstance(masks, torch.Tensor):
        masks = masks.cpu().numpy()
    if isinstance(iou_scores, torch.Tensor):
        iou_scores = iou_scores.cpu().numpy()

    return masks.astype(bool), iou_scores

In [ ]:
def clean_and_fill_mask(mask: np.ndarray) -> np.ndarray:
    """
    Use convex hull to fill gaps (neck/arm openings).
    Clothing items are roughly convex, so this works well.
    """
    hull = convex_hull_image(mask)
    return hull.astype(bool)


def extract_with_filled_background(
    image_np: np.ndarray,
    original_mask: np.ndarray,
    filled_mask: np.ndarray,
) -> np.ndarray:
    """
    - Original mask pixels -> real image (actual fabric)
    - Hull-added pixels -> dominant fabric color fill
    - Everything else -> white background
    """
    fabric_pixels = image_np[original_mask]
    dominant_color = np.median(fabric_pixels, axis=0).astype(np.uint8)

    result = np.full_like(image_np, 255)              # white bg
    result[filled_mask] = dominant_color               # hull areas
    result[original_mask] = image_np[original_mask]    # real pixels

    return result


def extract_clothing(
    image_np: np.ndarray,
    original_mask: np.ndarray,
    filled_mask: np.ndarray,
    save_transparent: bool = True,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[tuple]]:
    """
    Extract clothing region using the masks, crop tightly.

    Returns:
        extracted_white: (H, W, 3) clothing on white bg, cropped
        extracted_rgba: (H, W, 4) transparent bg, cropped (or None)
        bbox: (x1, y1, x2, y2)
    """
    rows = np.any(filled_mask, axis=1)
    cols = np.any(filled_mask, axis=0)

    if not rows.any() or not cols.any():
        return None, None, None

    y1, y2 = np.where(rows)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]

    pad = 10
    y1 = max(0, y1 - pad)
    y2 = min(filled_mask.shape[0], y2 + pad)
    x1 = max(0, x1 - pad)
    x2 = min(filled_mask.shape[1], x2 + pad)

    # White background version (with convex hull fill)
    white_bg = extract_with_filled_background(image_np, original_mask, filled_mask)
    cropped_white = white_bg[y1:y2, x1:x2]

    # Transparent background version
    cropped_rgba = None
    if save_transparent:
        rgba = np.zeros((image_np.shape[0], image_np.shape[1], 4), dtype=np.uint8)
        rgba[:, :, :3] = image_np
        rgba[:, :, 3] = (filled_mask * 255).astype(np.uint8)
        cropped_rgba = rgba[y1:y2, x1:x2]

    return cropped_white, cropped_rgba, (x1, y1, x2, y2)

---
## 6. Visualization Helpers

In [ ]:
def visualize_detections(image_np, boxes, masks, labels, scores):
    """Create an annotated visualization using the supervision library."""
    detections = sv.Detections(
        xyxy=boxes,
        mask=masks if len(masks) > 0 else None,
        confidence=scores,
    )

    annotated = image_np.copy()

    if masks is not None and len(masks) > 0:
        mask_annotator = sv.MaskAnnotator(opacity=0.4, color_lookup=sv.ColorLookup.INDEX)
        annotated = mask_annotator.annotate(annotated, detections)

    box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.INDEX)
    annotated = box_annotator.annotate(annotated, detections)

    label_strs = [f"{label}: {score:.2f}" for label, score in zip(labels, scores)]
    label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.INDEX)
    annotated = label_annotator.annotate(annotated, detections, labels=label_strs)

    return annotated


def show_pipeline_result(image_path, text_prompt, models, config):
    """Visualize the full pipeline on a single image inline."""
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    boxes, scores, labels = detect_clothing(image, text_prompt, models, config)
    print(f"Detected {len(boxes)} items: {labels}")
    print(f"Scores: {scores.round(3) if len(scores) > 0 else 'N/A'}")

    if len(boxes) == 0:
        plt.figure(figsize=(8, 8))
        plt.imshow(image_np)
        plt.title("No detections")
        plt.axis('off')
        plt.show()
        return

    masks, iou_scores = segment_with_sam2(image_np, boxes, models)
    original_masks = masks.copy()
    filled_masks = np.array([clean_and_fill_mask(m) for m in masks])

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    # Original + boxes
    axes[0].imshow(image_np)
    for box, label, score in zip(boxes, labels, scores):
        bx1, by1, bx2, by2 = box.astype(int)
        rect = plt.Rectangle((bx1, by1), bx2-bx1, by2-by1, fill=False, color='lime', linewidth=2)
        axes[0].add_patch(rect)
        axes[0].text(bx1, by1-5, f"{label} {score:.2f}", color='lime', fontsize=10,
                     bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
    axes[0].set_title("Detection (Grounding DINO)")
    axes[0].axis('off')

    # Mask overlay
    combined = filled_masks.any(axis=0)
    overlay = image_np.copy()
    overlay[combined] = (overlay[combined] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
    axes[1].imshow(overlay)
    axes[1].set_title("Segmentation (SAM 2 + hull fill)")
    axes[1].axis('off')

    # Extracted on white
    white_bg, _, _ = extract_clothing(image_np, original_masks[0], filled_masks[0], save_transparent=False)
    if white_bg is not None:
        axes[2].imshow(white_bg)
    axes[2].set_title("Extracted Clothing")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

---
## 7. Single Image Test

Upload or point to a test image and verify the pipeline works before batch processing.

In [ ]:
# Option A: Upload a test image
# from google.colab import files
# uploaded = files.upload()
# test_path = "/content/" + list(uploaded.keys())[0]

# Option B: Use an image from your dataset
test_path = "/content/drive/MyDrive/fashion_dataset/tshirts/0_0.jpg"  # <-- adjust
test_prompt = "t-shirt. crew neck top."

show_pipeline_result(test_path, test_prompt, models, config)

---
## 8. Threshold Tuning

Use this to find the best `BOX_THRESHOLD` for each category before batch processing.

| Clothing Type | Suggested `BOX_THRESHOLD` | Notes |
|---|---|---|
| Simple (t-shirts, shirts, jeans, shorts) | 0.30 | Standard |
| Complex (sarees, lehenga, dupatta) | 0.20 | Draped/unusual shapes |
| Layered (blazers over shirts, jackets) | 0.25 | May detect overlapping items |
| Ethnic (kurtas, salwar, ethnic tops) | 0.25 | Prompt engineering matters |

In [ ]:
def threshold_sweep(image_path, text_prompt, thresholds=None):
    """Test different detection thresholds on a single image."""
    if thresholds is None:
        thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

    image = Image.open(image_path).convert("RGB")
    inputs = models['gdino_processor'](
        images=image, text=text_prompt, return_tensors="pt"
    ).to(config.DEVICE)

    with torch.no_grad():
        outputs = models['gdino_model'](**inputs)

    results = models['gdino_processor'].post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        target_sizes=[image.size[::-1]],
    )[0]

    all_scores = results["scores"].cpu()
    all_labels = results["labels"]

    print(f"Image: {image_path}")
    print(f"Prompt: '{text_prompt}'")
    print(f"{'Threshold':<12} {'Detections':<12} {'Labels'}")
    print("-" * 60)

    for t in thresholds:
        keep = all_scores >= t
        n = keep.sum().item()
        lbls = [l for l, k in zip(all_labels, keep) if k]
        print(f"{t:<12.2f} {n:<12} {lbls}")


# threshold_sweep("/content/drive/MyDrive/fashion_dataset/tshirts/image_0.jpg", "t-shirt. crew neck top.")

---
## 9. Batch Processing

Process the entire dataset. Images are read directly from each category folder (flat structure).

```
DATASET_ROOT/
  {category}/
      image_0.jpg, image_1.jpg, ...
      description.txt
```

In [ ]:
def process_single_image(
    image_path: str,
    text_prompt: str,
    models: dict,
    config: Config,
    output_dir: str = None,
) -> list:
    """
    Full pipeline for a single image:
    detect -> segment -> clean mask -> extract -> save.
    """
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    # Detect
    boxes, scores, labels = detect_clothing(image, text_prompt, models, config)
    if len(boxes) == 0:
        return []

    # Segment
    masks, iou_scores = segment_with_sam2(image_np, boxes, models)
    if len(masks) == 0:
        return []

    # Clean masks with convex hull
    original_masks = masks.copy()
    filled_masks = np.array([clean_and_fill_mask(m) for m in masks])

    # Extract each detected item
    results = []
    for i, (orig_mask, fill_mask, label, det_score, iou_score) in enumerate(
        zip(original_masks, filled_masks, labels, scores, iou_scores)
    ):
        extracted_white, extracted_rgba, bbox = extract_clothing(
            image_np, orig_mask, fill_mask,
            save_transparent=config.SAVE_TRANSPARENT,
        )
        if extracted_white is None:
            continue

        result = {
            'mask': fill_mask,
            'extracted_white': extracted_white,
            'extracted_rgba': extracted_rgba,
            'bbox': bbox,
            'label': label,
            'detection_score': float(det_score),
            'iou_score': float(iou_score),
        }
        results.append(result)

        # Save outputs
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            stem = Path(image_path).stem

            Image.fromarray(extracted_white).save(
                os.path.join(output_dir, f"{stem}_seg_{i}.jpg"), quality=95
            )
            if extracted_rgba is not None:
                Image.fromarray(extracted_rgba).save(
                    os.path.join(output_dir, f"{stem}_seg_{i}.png")
                )
            if config.SAVE_MASK:
                Image.fromarray((fill_mask * 255).astype(np.uint8)).save(
                    os.path.join(output_dir, f"{stem}_seg_{i}_mask.png")
                )

    # Save visualization
    if output_dir and config.SAVE_VISUALIZATION and results:
        vis = visualize_detections(image_np, boxes, filled_masks, labels, scores)
        Image.fromarray(vis).save(
            os.path.join(output_dir, f"{Path(image_path).stem}_annotated.jpg"), quality=90
        )

    return results

In [ ]:
def process_dataset(
    config: Config,
    models: dict,
    categories: dict = None,
    box_threshold_override: float = None,
) -> dict:
    """
    Process the full dataset (flat structure: images directly in category folders).

    Expected input:
        DATASET_ROOT/{category}/image_0.jpg, image_1.jpg, ..., description.txt

    Output:
        OUTPUT_ROOT/{category}/image_0_seg_0.jpg, ..., metadata.json
    """
    if categories is None:
        categories = config.ALL_CATEGORIES

    # Optionally override threshold for this batch
    original_threshold = config.BOX_THRESHOLD
    if box_threshold_override is not None:
        config.BOX_THRESHOLD = box_threshold_override

    dataset_root = Path(config.DATASET_ROOT)
    output_root = Path(config.OUTPUT_ROOT)

    stats = {
        'total_images': 0,
        'total_detections': 0,
        'failed_images': 0,
        'categories_processed': {},
    }

    for category, text_prompt in categories.items():
        category_dir = dataset_root / category

        if not category_dir.exists():
            print(f"Skipping {category} - directory not found")
            continue

        print(f"\n{'='*60}")
        print(f"Processing category: {category}")
        print(f"Text prompt: '{text_prompt}'")
        print(f"Box threshold: {config.BOX_THRESHOLD}")
        print(f"{'='*60}")

        cat_detections = 0
        cat_images = 0

        # Collect all images directly in the category folder
        image_files = sorted(
            list(category_dir.glob("*.jpg")) +
            list(category_dir.glob("*.jpeg")) +
            list(category_dir.glob("*.png"))
        )

        if not image_files:
            print(f"  No images found in {category_dir}")
            continue

        output_dir = output_root / category
        all_metadata = []

        for img_path in tqdm(image_files, desc=f"  {category}"):
            cat_images += 1
            try:
                results = process_single_image(
                    str(img_path), text_prompt, models, config,
                    output_dir=str(output_dir),
                )
                cat_detections += len(results)

                for r in results:
                    all_metadata.append({
                        'image': img_path.name,
                        'label': r['label'],
                        'detection_score': r['detection_score'],
                        'iou_score': r['iou_score'],
                        'bbox': [int(x) for x in r['bbox']],
                    })

            except Exception as e:
                print(f"  ERROR processing {img_path.name}: {e}")
                stats['failed_images'] += 1

        # Save aggregated metadata for the category
        if all_metadata:
            os.makedirs(output_dir, exist_ok=True)
            with open(output_dir / "metadata.json", 'w') as f:
                json.dump(all_metadata, f, indent=2)

        stats['total_images'] += cat_images
        stats['total_detections'] += cat_detections
        stats['categories_processed'][category] = {
            'images': cat_images,
            'detections': cat_detections,
        }

        print(f"  -> {category}: {cat_images} images, {cat_detections} detections")

    # Restore threshold
    config.BOX_THRESHOLD = original_threshold

    # Summary
    print(f"\n{'='*60}")
    print("DATASET PROCESSING COMPLETE")
    print(f"{'='*60}")
    print(f"Total images processed: {stats['total_images']}")
    print(f"Total clothing items extracted: {stats['total_detections']}")
    print(f"Failed images: {stats['failed_images']}")
    print(f"Output saved to: {output_root}")

    # Save stats
    os.makedirs(output_root, exist_ok=True)
    with open(output_root / "processing_stats.json", 'w') as f:
        json.dump(stats, f, indent=2)

    return stats

In [ ]:
# Process men's clothing (higher threshold)
stats_mens = process_dataset(config, models,
                             categories=config.MENS_CLOTHING,
                             box_threshold_override=0.30)

# Process women's clothing (lower threshold for draped/ethnic wear)
stats_womens = process_dataset(config, models,
                               categories=config.WOMENS_CLOTHING,
                               box_threshold_override=0.20)

---
## 10. Quality Check

Visually inspect a sample of segmented outputs.

In [ ]:
def quality_check(category: str, num_samples: int = 12):
    """Display a grid of segmented images for visual QA."""
    seg_dir = Path(config.OUTPUT_ROOT) / category
    images = sorted(list(seg_dir.glob("*_seg_0.jpg")))[:num_samples]

    if not images:
        print(f"No segmented images found for {category}")
        return

    cols = 4
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else list(axes)
    else:
        axes = axes.flatten()

    for ax, img_path in zip(axes, images):
        img = Image.open(img_path)
        ax.imshow(np.array(img))
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    for ax in axes[len(images):]:
        ax.axis('off')

    plt.suptitle(f"Quality Check: {category}", fontsize=14)
    plt.tight_layout()
    plt.show()


# Check a few categories
#quality_check("tshirts")
quality_check("jeans")
#quality_check("saree")

---
## 11. Build LoRA Training Dataset

Prepare segmented outputs for Flux LoRA fine-tuning (kohya_ss / ai-toolkit / SimpleTuner format).

```
lora_training_data/
  img/
    000001.png
    000001.txt
    000002.png
    000002.txt
    ...
  metadata.csv
```

In [ ]:
def clean_caption_for_training(caption: str, category: str) -> str:
    """
    Clean Myntra product descriptions into concise training captions.

    Good: "navy blue solid round neck short sleeve cotton t-shirt"
    Bad:  "Buy Men Navy Blue Solid Round Neck T-shirt - Tshirts for Men 12345 | Myntra"
    """
    caption = caption.lower().strip()

    noise_patterns = [
        r'buy\s+(men|women|boys|girls)\s+',
        r'\s*-\s*(tshirts|shirts|jeans|tops|dresses)\s+for\s+(men|women)\s+\d+.*',
        r'\|\s*myntra.*$',
        r'free shipping.*$',
        r'cash on delivery.*$',
        r'(?:rs\.?|inr|\u20b9)\s*\d+',
        r'(?:extra\s+)?\d+%\s*off',
        r'best\s+price.*$',
    ]

    for pattern in noise_patterns:
        caption = re.sub(pattern, '', caption, flags=re.IGNORECASE)

    caption = re.sub(r'\s+', ' ', caption).strip()

    if not caption.startswith(('a ', 'an ')):
        caption = f"a {caption}"

    return caption


def build_lora_dataset(config: Config) -> list:
    """
    Build a LoRA training dataset from segmented outputs.

    Reads description.txt from each category folder in the *input* dataset
    (DATASET_ROOT/{category}/description.txt) and pairs it with segmented images.
    """
    output_root = Path(config.OUTPUT_ROOT)
    dataset_root = Path(config.DATASET_ROOT)
    lora_dir = output_root / "lora_training_data" / "img"
    os.makedirs(lora_dir, exist_ok=True)

    csv_path = output_root / "lora_training_data" / "metadata.csv"

    idx = 0
    records = []

    for category in config.ALL_CATEGORIES.keys():
        cat_output_dir = output_root / category
        cat_input_dir = dataset_root / category

        if not cat_output_dir.exists():
            continue

        # Load description from the input category folder
        base_caption = ""
        desc_txt = cat_input_dir / "description.txt"
        if desc_txt.exists():
            base_caption = desc_txt.read_text().strip()

        # Find all white-bg segmented images
        seg_images = sorted(cat_output_dir.glob("*_seg_*.jpg"))

        for img_path in seg_images:
            caption = base_caption if base_caption else ""

            if not caption:
                caption = f"a {category.replace('-', ' ')} clothing item"
            else:
                caption = clean_caption_for_training(caption, category)

            new_img_name = f"{idx:06d}.png"
            new_txt_name = f"{idx:06d}.txt"

            img = Image.open(img_path).convert("RGB")
            img.save(lora_dir / new_img_name)
            (lora_dir / new_txt_name).write_text(caption)

            records.append({
                'index': idx,
                'image': new_img_name,
                'caption': caption,
                'category': category,
                'source_path': str(img_path),
            })

            idx += 1

    # Save CSV
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['index', 'image', 'caption', 'category', 'source_path'])
        writer.writeheader()
        writer.writerows(records)

    print(f"LoRA training dataset built: {idx} images")
    print(f"  Images: {lora_dir}")
    print(f"  Metadata: {csv_path}")

    return records

In [ ]:
records = build_lora_dataset(config)

print(f"\nPipeline complete!")
print("Next steps:")
print("  1. Review segmented outputs for quality")
print("  2. Remove any bad segmentations manually")
print("  3. Use the lora_training_data/ folder with kohya_ss or ai-toolkit")